## Import Libraries

In [1]:
# imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)

Pandas version: 3.0.5
Numpy version: 2.5.1


## Load Dataset and Make Inspection

In [2]:

# Load raw dataset
df = pd.read_csv("../data/raw/synthetic_loan_approval_dataset.csv")

# Inspect 
print(df.head(5))
df.info()


  Application_ID  Age  Gender Marital_Status      Education  Dependents  \
0      APP100000   24  Female         Single       Graduate           0   
1      APP100001   26    Male        Married       Graduate           2   
2      APP100002   58  Female        Married  Post Graduate           1   
3      APP100003   38  Others        Married       Graduate           1   
4      APP100004   32  Female         Single            PhD           0   

  Residence_Type  City_Tier Employment_Type  Years_at_Current_Job  ...  \
0          Rural          3         Private                     1  ...   
1          Urban          1         Private                     0  ...   
2          Urban          1      Government                    21  ...   
3          Rural          4         Private                     2  ...   
4          Rural          4      Government                     1  ...   

   Aadhaar_Verified  Loan_Purpose  Loan_Amount  Loan_Tenure  Interest_Rate  \
0               Yes       

## Feature Selection & Dropping Redundant Columns



In this step, we drop specific columns from the dataset to refine our feature set for the decision tree model:

* **`Application_ID`:** Unique identifier; contains no predictive signal.
* **`Gender`:** Dropped to prevent demographic bias and maintain ethical lending standards.
* **`Annual_Income`:** Excluded because we have monthly income already which can be used to compute annual, no new information provided by it.
* **`PAN_Verified` & `Aadhaar_Verified`:** Verification flags that do not directly measure credit risk.
* **`Property_Value`:** Redundant, as collateral risk is already captured by `Loan_Amount` and `Loan_to_Value` (LTV).

In [3]:

# Drop unwanted columns
columns_to_drop = [
    "Application_ID",
    "Gender",
    "Annual_Income",
    "PAN_Verified",
    "Aadhaar_Verified",
    "Property_Value"
]

df.drop(columns=columns_to_drop, inplace=True)

In [4]:
# verifying dropped columns
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      20000 non-null  int64  
 1   Marital_Status           20000 non-null  str    
 2   Education                20000 non-null  str    
 3   Dependents               20000 non-null  int64  
 4   Residence_Type           20000 non-null  str    
 5   City_Tier                20000 non-null  int64  
 6   Employment_Type          20000 non-null  str    
 7   Years_at_Current_Job     20000 non-null  int64  
 8   Total_Work_Experience    20000 non-null  int64  
 9   Monthly_Income           20000 non-null  int64  
 10  Other_Income             19587 non-null  float64
 11  Existing_Loans           20000 non-null  int64  
 12  Existing_Loan_Amount     20000 non-null  int64  
 13  Monthly_EMI              20000 non-null  int64  
 14  Debt_to_Income           20000 no

## Define Features ($X$) and Target ($y$)



Before splitting the dataset for model training, we separate our dataset into the feature matrix ($X$) and the target vector ($y$).

We separate the dataset into X and y to tell the machine learning algorithm which data are the input features (X) and which data are the target/label (y) that we want the model to predict.

* **Feature Matrix ($X$):** Contains all predictor variables (both raw numerical metrics and engineered features) while excluding the target column.
* **Target Vector ($y$):** Contains the ground-truth `Loan_Status` labels (`Approved` and `Rejected`). The classifier accepts these labels directly.

In [5]:
# Define features (X) and target (y)

X = df.drop("Loan_Status", axis=1)
y = df["Loan_Status"]

# Verify the results
print("=========================================")
print("X AND y VERIFICATION")
print("=========================================")

print("\nShape of X:", X.shape)
print("Shape of y:", y.shape)

print("\nFeatures in X:")
print(X.columns.tolist())

print("\nTarget column:")
print(y.name)

X AND y VERIFICATION

Shape of X: (20000, 32)
Shape of y: (20000,)

Features in X:
['Age', 'Marital_Status', 'Education', 'Dependents', 'Residence_Type', 'City_Tier', 'Employment_Type', 'Years_at_Current_Job', 'Total_Work_Experience', 'Monthly_Income', 'Other_Income', 'Existing_Loans', 'Existing_Loan_Amount', 'Monthly_EMI', 'Debt_to_Income', 'Savings', 'Investments', 'Bank_Balance', 'Credit_Card_Utilization', 'Number_of_Bank_Accounts', 'Number_of_Credit_Cards', 'Credit_Score', 'Loan_Defaults', 'Missed_Payments', 'Tax_Return_Filed', 'Loan_Purpose', 'Loan_Amount', 'Loan_Tenure', 'Interest_Rate', 'Collateral', 'Collateral_Value', 'Loan_to_Value']

Target column:
Loan_Status


### 📊 Dataset Separation Verification

The features and target variable have been successfully partitioned:

* **Predictor Matrix ($X$):** Consists of $32$ input features across $20,000$ samples.
* **Target Variable ($y$):** Consists of $20,000$ `Loan_Status` labels.

> **Next Step:** With $X$ and $y$ cleanly defined, we are ready to perform a stratified train-test split to evaluate our model's performance on unseen data.

## Train/Test Split

### ✂️ Train-Test Partitioning

We split our feature matrix ($X$) and target vector ($y$) into training and testing subsets to evaluate how well our model generalizes to unseen data.

* **Split Ratio ($80/20$):** $80\%$ of the data is allocated for model training, leaving $20\%$ as an independent test holdout.


In [6]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Verify the split
print("=========================================")
print("TRAIN / TEST SPLIT VERIFICATION")
print("=========================================")

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining percentage:",
      round(len(X_train) / len(X) * 100, 2), "%")

print("Testing percentage:",
      round(len(X_test) / len(X) * 100, 2), "%")

TRAIN / TEST SPLIT VERIFICATION

X_train shape: (16000, 32)
X_test shape: (4000, 32)

y_train shape: (16000,)
y_test shape: (4000,)

Training percentage: 80.0 %
Testing percentage: 20.0 %


# Exploratory Data Analysis (EDA)


---

## Data Cleaning: Missing Value Imputation

The training set is now the reference dataset for all preprocessing decisions:

1. **Inspect `X_train`:** Identify missing values and study the training-data distribution.
2. **Fit on `X_train`:** Calculate learned values, such as medians, using only training data.
3. **Transform both splits:** Apply the same training-derived values to `X_train` and `X_test`.
4. **Verify both splits:** Confirm that no null entries remain before model training.

This prevents the test holdout from influencing the data preparation process.
---

In [7]:
# Inspect missing values in X_train only.
# The test set remains unseen while preprocessing decisions are made.
missing_values = X_train.isnull().sum()
print("\nMissing values in X_train:\n", missing_values)

print('\n')
# Checking if there is missing Values in Target Leabel
missing_values_y_train = y_train.isnull().sum()
print("\nMissing values in y_train:\n", missing_values_y_train)



Missing values in X_train:
 Age                           0
Marital_Status                0
Education                     0
Dependents                    0
Residence_Type                0
City_Tier                     0
Employment_Type               0
Years_at_Current_Job          0
Total_Work_Experience         0
Monthly_Income                0
Other_Income                336
Existing_Loans                0
Existing_Loan_Amount          0
Monthly_EMI                   0
Debt_to_Income                0
Savings                       0
Investments                 322
Bank_Balance                154
Credit_Card_Utilization       0
Number_of_Bank_Accounts       0
Number_of_Credit_Cards        0
Credit_Score                 99
Loan_Defaults                 0
Missed_Payments               0
Tax_Return_Filed            136
Loan_Purpose                  0
Loan_Amount                   0
Loan_Tenure                   0
Interest_Rate                99
Collateral                    0
Collateral_

### Summary of Identified Missing (`NA`) Values

The missingness report above is calculated from **`X_train` only**. This is the correct dataset to use when choosing and fitting preprocessing values.

| Feature Type | Strategy | Leakage-Safe Rule |
| :--- | :--- | :--- |
| Numerical features | Median or zero-fill, according to domain meaning | Calculate medians from `X_train` only. |
| Categorical features | Explicit `Unknown` category | Apply the same fill value to both splits. |

> **Key Observation:** `Loan_to_Value` and `Collateral_Value` have substantial missingness. Their imputation values are learned from the training subset and reused unchanged for the test subset.

### Missing Value Handling Strategy

Each transformation below follows a fit/transform workflow:

| Feature Column | Strategy | Training Fit and Test Transform Rule |
| :--- | :--- | :--- |
| **`Loan_to_Value`** | Median imputation | Calculate the median from `X_train`, then fill both splits with it. |
| **`Collateral_Value`** | Median imputation | Calculate the median from `X_train`, then fill both splits with it. |
| **`Other_Income`** | Zero-fill | Apply the fixed domain value (`0`) to both splits. |
| **`Investments`** | Zero-fill | Apply the fixed domain value (`0`) to both splits. |
| **`Bank_Balance`** | Median imputation | Fit median on `X_train`; reuse on `X_test`. |
| **`Tax_Return_Filed`** | Constant `Unknown` | Apply the fixed category to both splits. |
| **`Credit_Score`** | Median imputation | Fit median on `X_train`; reuse on `X_test`. |
| **`Interest_Rate`** | Median imputation | Fit median on `X_train`; reuse on `X_test`. |

> **Important:** No median, category list, or other learned value is calculated from `X_test`.

### Missing Data: Feature 01 — Evaluating `Loan_to_Value`

In [8]:
print("=== Loan_to_Value Distribution ===")
print(X_train['Loan_to_Value'].describe())

print("\n=== Loan_to_Value Quantiles ===")
print(X_train['Loan_to_Value'].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]))

print("\n=== Loan_to_Value Missingness ===")
print(X_train['Loan_to_Value'].isna().sum())
print("Percentage missing:", X_train['Loan_to_Value'].isna().mean() * 100)



=== Loan_to_Value Distribution ===
count    10061.000000
mean        67.667578
std         12.657534
min         45.600000
25%         57.260000
50%         65.750000
75%         77.090000
max         97.941706
Name: Loan_to_Value, dtype: float64

=== Loan_to_Value Quantiles ===
0.01    47.386
0.05    50.120
0.25    57.260
0.50    65.750
0.75    77.090
0.95    91.330
0.99    95.000
Name: Loan_to_Value, dtype: float64

=== Loan_to_Value Missingness ===
5939
Percentage missing: 37.11875


### Statistical Analysis: Why Median Imputation for `Loan_to_Value`?

By inspecting the summary statistics and quantile distributions for **`Loan_to_Value` (LTV)**, we observe key distribution traits that dictate our imputation strategy:

#### Key Statistical Findings:
* **Right-Skewed Distribution:** The mean ($\approx 67.67\%$) is higher than the median / 50th percentile ($65.75\%$).
* **Outlier / Tail Impact:** The top percentile reaches **97.94%**, pulling the mean upward due to high-risk upper-bound loans.
* **Stable Central Range:** The Interquartile Range (IQR) spans **57.26% to 77.09%**, showing that typical loan ratios cluster predictably around 65.75%.

> **Conclusion & Decision:**  
> Because **37.12% (5,939 rows)** of the training data is missing, using the **Mean** would systematically overestimate risk by shifting imputed values toward high extreme ratios. The **Median (65.75%)** represents the true 50th percentile and is robust against tail-end skewness.

### Missing Data: Feature 02 — Evaluating `Collateral_Value`

In [9]:
print("=== Collateral_Value Missingness ===")

# Count missing values
missing_count = X_train['Collateral_Value'].isna().sum()
# Calculate percentage missing
missing_percentage = X_train['Collateral_Value'].isna().mean() * 100

print("Number of missing values:", missing_count)
print("Percentage missing:", round(missing_percentage, 2), "%")


print("\n=== Collateral_Value Distribution ===")

# Display descriptive statistics for available values
print(X_train['Collateral_Value'].describe())


print("\n=== Collateral_Value Quantiles ===")

# Examine the distribution at different percentiles
print(
    X_train['Collateral_Value'].quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

=== Collateral_Value Missingness ===
Number of missing values: 5875
Percentage missing: 36.72 %

=== Collateral_Value Distribution ===
count    1.012500e+04
mean     4.031651e+06
std      3.899838e+06
min      7.238000e+03
25%      1.282284e+06
50%      3.154642e+06
75%      5.708916e+06
max      1.220902e+08
Name: Collateral_Value, dtype: float64

=== Collateral_Value Quantiles ===
0.01      165803.36
0.05      397409.80
0.25     1282284.00
0.50     3154642.00
0.75     5708916.00
0.95    10482391.20
0.99    15311821.20
Name: Collateral_Value, dtype: float64


### Missing Data Assessment: `Collateral_Value`

Evaluating central metrics to determine the safest imputation method for **5,875 missing rows (36.72%)** in the training data:

| Metric | Formatted Value | Impact on Imputation Choice |
| :--- | :--- | :--- |
| **Mean** | `4,031,651` | **Heavily Inflated:** Pulled upward by high-end outliers (max: 122M). |
| **Median (50%)** | `3,154,642` | **Robust Baseline:** Unaffected by upper-tail extremes. |
| **IQR (25% – 75%)** | `1,282,284 – 5,708,916` | Demonstrates typical middle-50% collateral spread. |

**Decision:** We choose **Median Imputation (`3,154,642`)** to prevent high-value properties from artificially inflating credit collateral estimates across missing records.

### Strategies fo Remaining Missing Values



The remaining six features are imputed using domain logic (zero-fill, explicit category) or robust central tendencies (median):

| Feature | Missing Strategy | Rationale |
| :--- | :--- | :--- |
| **`Other_Income`** | Zero-Fill (`0`) | Absence of recorded value indicates no secondary income stream. |
| **`Investments`** | Zero-Fill (`0`) | Absence of recorded value indicates no reported investment holdings. |
| **`Bank_Balance`** | Median Imputation | Robust to skewness caused by high-balance account outliers. |
| **`Tax_Return_Filed`** | Constant (`'Unknown'`) | Preserves missingness explicitly as a distinct categorical flag. |
| **`Credit_Score`** | Median Imputation | Protects against tail-end credit extremes. |
| **`Interest_Rate`** | Median Imputation | Provides a stable central rate un-skewed by subprime interest outliers. |

## Identify Numerical Features for Outlier Analysis

In [10]:


numerical_columns = X_train.select_dtypes(include=['number']).columns.tolist()

print("=== Numerical Features ===")
print(numerical_columns)

print("\nNumber of numerical features:", len(numerical_columns))

=== Numerical Features ===
['Age', 'Dependents', 'City_Tier', 'Years_at_Current_Job', 'Total_Work_Experience', 'Monthly_Income', 'Other_Income', 'Existing_Loans', 'Existing_Loan_Amount', 'Monthly_EMI', 'Debt_to_Income', 'Savings', 'Investments', 'Bank_Balance', 'Credit_Card_Utilization', 'Number_of_Bank_Accounts', 'Number_of_Credit_Cards', 'Credit_Score', 'Loan_Defaults', 'Missed_Payments', 'Loan_Amount', 'Loan_Tenure', 'Interest_Rate', 'Collateral_Value', 'Loan_to_Value']

Number of numerical features: 25


### Outlier Analysis: Interquartile Range (IQR)

The **Interquartile Range (IQR)** is a statistical method used to identify potential outliers in numerical data. It is calculated as the difference between the third quartile ($Q_3$) and the first quartile ($Q_1$):

$$\text{IQR} = Q_3 - Q_1$$

In this study, a value is considered an outlier if it meets either condition:

* **Lower Outlier Boundary:** $Q_1 - 1.5(\text{IQR})$
* **Upper Outlier Boundary:** $Q_3 + 1.5(\text{IQR})$

This method will be applied to the numerical features in the dataset to identify unusually high or low values that may affect the performance of the machine learning model.

### Outlier Detection Example using IQR

Given the dataset:
`10, 12, 13, 14, 15, 16, 17, 18, 20, 50`

Visually, **50** appears unusually high. We can use the Interquartile Range (IQR) method to mathematically verify if it is an outlier.

---

#### Step 1: Find $Q_1$ and $Q_3$

1. **Find the Median ($Q_2$):**  
   With $N = 10$ values, the median lies between the 5th and 6th positions:
   $$Q_2 = \frac{15 + 16}{2} = 15.5$$

2. **Split the Data:**
   * **Lower Half:** `10, 12, 13, 14, 15`
   * **Upper Half:** `16, 17, 18, 20, 50`

3. **Calculate Quartiles & IQR:**
   * $Q_1 = 13$ (median of lower half)
   * $Q_3 = 18$ (median of upper half)
   * $\text{IQR} = Q_3 - Q_1 = 18 - 13 = 5$

---

#### Step 2: Calculate the Outlier Boundaries

* **Lower Boundary:** $Q_1 - 1.5(\text{IQR}) = 13 - 1.5(5) = 5.5$
* **Upper Boundary:** $Q_3 + 1.5(\text{IQR}) = 18 + 1.5(5) = 25.5$

Acceptable values fall within **5.5 to 25.5**.

---

#### Step 3: Identify Outliers

Evaluating our values against the boundaries $[5.5, 25.5]$:

* All values from 10 to 20 fall within bounds.
* **$50 > 25.5$** $\rightarrow$ **50 is an outlier.**

### DETECT POTENTIAL OUTLIERS IN NUMERICAL FEATURES

In [11]:


# Create a list to store outlier results
outlier_results = []

# Calculate IQR-based outliers for each numerical feature
for column in numerical_columns:
    
    # Calculate Q1 and Q3
    Q1 = X_train[column].quantile(0.25)
    Q3 = X_train[column].quantile(0.75)
    
    # Calculate the Interquartile Range
    IQR = Q3 - Q1
    
    # Calculate lower and upper bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Count potential outliers
    outlier_count = (
        (X_train[column] < lower_bound) |
        (X_train[column] > upper_bound)
    ).sum()
    
    # Calculate percentage of potential outliers
    outlier_percentage = (outlier_count / len(X_train)) * 100
    
    # Store the results
    outlier_results.append({
        'Feature': column,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower_Bound': lower_bound,
        'Upper_Bound': upper_bound,
        'Outlier_Count': outlier_count,
        'Outlier_Percentage': outlier_percentage
    })

# Convert results to a DataFrame
outlier_summary = pd.DataFrame(outlier_results)

# Sort from highest to lowest number of potential outliers
outlier_summary = outlier_summary.sort_values(
    by='Outlier_Count',
    ascending=False
)

print("=== Potential Outlier Summary ===")
print(outlier_summary.to_string(index=False))

=== Potential Outlier Summary ===
                Feature         Q1          Q3         IQR   Lower_Bound  Upper_Bound  Outlier_Count  Outlier_Percentage
        Missed_Payments       0.00       1.000       1.000 -1.500000e+00 2.500000e+00           2200            13.75000
           Other_Income       0.00   16473.000   16473.000 -2.470950e+04 4.118250e+04           2142            13.38750
          Loan_Defaults       0.00       0.000       0.000  0.000000e+00 0.000000e+00           1979            12.36875
            Investments       0.00  279958.250  279958.250 -4.199374e+05 6.998956e+05           1359             8.49375
           Credit_Score     571.00     670.000      99.000  4.225000e+02 8.185000e+02            869             5.43125
         Existing_Loans       0.00       1.000       1.000 -1.500000e+00 2.500000e+00            795             4.96875
           Bank_Balance  230371.50  811512.250  581140.750 -6.413396e+05 1.683223e+06            755             4.7187

### Feature Outlier Counts Based on output from above cell

| Feature | Outlier Count |
| :--- | :---: |
| **Missed_Payments** | 2,200 |
| **Other_Income** | 2,142 |
| **Loan_Defaults** | 1,979 |
| **Investments** | 1,359 |
| **Credit_Score** | 869 |
| **Existing_Loans** | 795 |
| **Bank_Balance** | 755 |
| **Years_at_Current_Job** | 736 |
| **Number_of_Bank_Accounts** | 604 |
| **Loan_Amount** | 571 |
| **Monthly_EMI** | 526 |
| **Savings** | 513 |
| **Existing_Loan_Amount** | 492 |
| **Total_Work_Experience** | 466 |
| **Number_of_Credit_Cards** | 439 |
| **Credit_Card_Utilization** | 377 |
| **Monthly_Income** | 284 |
| **Collateral_Value** | 275 |
| **Age** | 0 |
| **Dependents** | 0 |
| **City_Tier** | 0 |
| **Debt_to_Income** | 0 |
| **Interest_Rate** | 0 |
| **Loan_Tenure** | 0 |

## Outlier Treatment Classification Framework

> **Guiding Question:** *Does an extreme value represent a data error, or can it legitimately occur in real life?*

---

## 1. Classification Categories

* **⚪ Exclude from IQR (Count / Discrete Variables):** IQR is unsuitable for discrete or low-variance counts (often where $Q_1 = Q_3 = 0$).
* **🟢 Exclude from Removal (Legitimate Extreme Values):** Financial assets and income naturally follow long-tailed distributions; high values reflect real-world wealth, not data errors.
* **🟡 Investigate (Bounded / Logical Ranges):** Variables with hard physical, human, or mathematical limits that require domain validation.
* **🚫 Exclude (Categorical):** Numerically encoded ordinal categories where statistical distance has no mathematical meaning.
* **🔵 No Outliers Detected:** Zero outliers detected using standard IQR thresholds; no filtering needed.

---

## 2. Feature Classification Table

| Feature | Outlier Count | Classification | Primary Rationale |
| :--- | :---: | :---: | :--- |
| **Missed_Payments** | 2,200 | ⚪ Exclude | Discrete count; zero-inflated ($Q_1 = Q_3 = 0$). |
| **Other_Income** | 2,142 | 🟢 Exclude | Wealth variation is expected; do not trim top earners. |
| **Loan_to_Value (LTV)** | 0 | 🔵 No Outliers | No IQR outliers detected in the current training data. |
| **Loan_Defaults** | 1,979 | ⚪ Exclude | Discrete count; zero-inflated ($Q_1 = Q_3 = 0$). |
| **Collateral_Value** | 275 | 🟢 Exclude | Property values naturally vary across huge ranges. |
| **Investments** | 1,359 | 🟢 Exclude | High investment balances are valid assets. |
| **Credit_Score** | 869 | 🟡 Investigate | Must fall within fixed bureau bounds (e.g., $300–850$). |
| **Existing_Loans** | 795 | ⚪ Exclude | Discrete count; multiple loans are not inherently anomalous. |
| **Bank_Balance** | 755 | 🟢 Exclude | High liquidity is legitimate. |
| **Years_at_Current_Job** | 736 | 🟡 Investigate | Cap at total work experience / age plausibility. |
| **Number_of_Bank_Accounts** | 604 | ⚪ Exclude | Discrete count; 4–5 accounts is normal variation. |
| **Loan_Amount** | 571 | 🟢 Exclude | Large requests are valid business cases. |
| **Monthly_EMI** | 526 | 🟢 Exclude | Scales naturally with loan size, tenure, and rate. |
| **Savings** | 513 | 🟢 Exclude | Wealth accumulation varies dramatically. |
| **Existing_Loan_Amount** | 492 | 🟢 Exclude | High debt burdens legitimately exist. |
| **Total_Work_Experience** | 466 | 🟡 Investigate | Check against age and realistic career duration. |
| **Number_of_Credit_Cards** | 439 | ⚪ Exclude | Discrete count; multiple cards are valid. |
| **Credit_Card_Utilization** | 377 | 🟡 Investigate | Percentage bounded ratio; check for values $> 100\%$. |
| **Monthly_Income** | 284 | 🟢 Exclude | High earners are legitimate; long-tailed distribution. |
| **Age** | 0 | 🔵 No Outliers | All values fall within standard IQR bounds ($Q_1 - 1.5\text{IQR}$ to $Q_3 + 1.5\text{IQR}$). |
| **Dependents** | 0 | 🔵 No Outliers | Low variance; all observed counts fall within standard boundaries. |
| **City_Tier** | 0 | 🔵 No Outliers / 🚫 Exclude | Ordinal categorical encoding; 0 outliers detected and math operations are invalid. |
| **Debt_to_Income (DTI)** | 0 | 🔵 No Outliers | Financial ratio; no extreme mathematical anomalies found in current data. |
| **Interest_Rate** | 0 | 🔵 No Outliers | All rate values reside within standard expected distributions. |
| **Loan_Tenure** | 0 | 🔵 No Outliers | Discrete time periods; all values conform to standard allowed loan terms. |

---

## 3.  Implementation Rules

1. **Never auto-drop 🟢 Green features:** Doing so skews model learning on high-value customers and high-volume transactions.
2. **Apply domain boundary checks on 🟡 Yellow features:** Replace IQR with logical constraints (e.g., $\text{Age} \le 100$, $\text{Utilization} \le 100\%$).
3. **Bypass ⚪ White and 🚫 Categorical features:** Handle via frequency encoding or tree-based splits instead of distance-based outlier filters.
4. **Pass-through 🔵 Blue features:** No action required during preprocessing as these features exhibit no statistical outliers.

### Domain Boundary Validation
  Checking whether the values in the dataset fall within the expected or realistic range for a particular field based on real-world rules and physical constraints.

### Features Requiring Investigation 
***Strategy: Domain Boundary Validation***

| Feature | Outlier Count | Classification | Primary Rationale & Domain Rules |
| :--- | :---: | :---: | :--- |
| **Loan_to_Value (LTV)** | 0 | 🔵 No IQR outliers | Domain validation still confirms the financial range is valid. |
| **Credit_Score** | 869 | 🟡 Investigate | Credit bureau score. Validate against standard range limits (e.g., must strictly fall within 300–850). |
| **Years_at_Current_Job** | 736 | 🟡 Investigate | Human employment limit. Cross-check against `Total_Work_Experience` and overall `Age` plausibility. |
| **Total_Work_Experience** | 466 | 🟡 Investigate | Career duration. Cross-check against `Age` (e.g., rule out $\text{Experience} > [\text{Age} - 16]$). |
| **Credit_Card_Utilization** | 466 | 🟡 Investigate | Bounded percentage ratio. Check for impossible negative values or flag extreme utilization over 100%. |

## Investigation 1: Domain Boundary Validation for Loan-to-Value (LTV)

### 💡 Critical Pause: Data Lineage & Formula Verification

Before jumping straight into **Domain Boundary Validation** for `Loan_to_Value` (LTV), I paused to ask an essential question: 

> *"Was `Loan_to_Value` actually calculated using its standard mathematical formula ($\frac{\text{Loan\_Amount}}{\text{Collateral\_Value}} \times 100$), or is there an underlying discrepancy in the data?"*

To verify this, I decided to run a quick manual calculation across sample rows to build a comparison table between the dataset's existing `Loan_to_Value` column and the formula output:

In [12]:
# ============================================================
# VERIFYING HOW Loan_to_Value WAS CALCULATED
# ============================================================

# Step 1: Select the columns we need
print("Step 1: Checking the relevant columns...")
print()

print(X_train[[
    "Loan_to_Value",
    "Loan_Amount",
    "Collateral_Value"
]].head(10))


# ============================================================
# Step 2: Calculate LTV ourselves using the assumed formula
# ============================================================

print("\nStep 2: Calculating LTV using:")
print("LTV = (Loan_Amount / Collateral_Value) * 100")
print()

calculated_ltv = (
    X_train["Loan_Amount"] / X_train["Collateral_Value"]
) * 100


# ============================================================
# Step 3: Create a comparison table
# ============================================================

comparison = X_train[[
    "Loan_to_Value",
    "Loan_Amount",
    "Collateral_Value"
]].copy()

comparison["Calculated_LTV"] = calculated_ltv


print("Step 3: Comparing the dataset's LTV with our calculated LTV...")
print()

print(comparison.head(10))



Step 1: Checking the relevant columns...

       Loan_to_Value  Loan_Amount  Collateral_Value
5164       82.990000      1396569         1682761.0
2385       68.990000       875653         1269277.0
2160       61.100000      1757797         2877057.0
16852      93.417104      1242243               NaN
13383      49.140000      3133013         6375059.0
18309            NaN      1632525               NaN
16378      76.090000       803223         1055615.0
14532            NaN      2687507               NaN
5787             NaN       492057               NaN
3695       89.930000      7000520         7783981.0

Step 2: Calculating LTV using:
LTV = (Loan_Amount / Collateral_Value) * 100

Step 3: Comparing the dataset's LTV with our calculated LTV...

       Loan_to_Value  Loan_Amount  Collateral_Value  Calculated_LTV
5164       82.990000      1396569         1682761.0       82.992713
2385       68.990000       875653         1269277.0       68.988330
2160       61.100000      1757797       

#### 🔍 Findings & Key Takeaway

* **The Observation:** While many rows matched my calculated values perfectly, some diverged. 
* **The Explanation:** The inspected rows include missing `Collateral_Value` or `Loan_to_Value` values. Where both source values are present, small differences are explained by rounding the stored LTV value to two decimal places; a ratio cannot be recalculated where collateral is missing.

#### 🧠 Personal Reflection
This step validated that the column structure was sound and ready for boundary checking, but more importantly, it reinforced an essential truth: **human judgment and critical thinking remain indispensable in the AI era**. 

While AI coding agents can execute scripts instantly, they won't automatically question data lineage or account for prior preprocessing choices like imputation. Your domain intuition as a data practitioner is just as important as the code itself.

### DOMAIN BOUNDARY VALIDATION FOR LOAN-TO-VALUE (LTV)

In [13]:


print("Checking Loan_to_Valubut i wound want e domain boundaries...")
print()

# Find LTV values outside the expected 0–100% range
ltv_invalid = X_train.loc[
    (X_train["Loan_to_Value"] <= 0) |
    (X_train["Loan_to_Value"] > 100),
    [
        "Loan_to_Value",
        "Loan_Amount",
        "Collateral_Value"
    ]
]

# Display the invalid/suspicious records
print("Potentially invalid LTV records:")
print(ltv_invalid)

# Count the number of violations
print("\nNumber of LTV domain violations:")
print(len(ltv_invalid))

Checking Loan_to_Valubut i wound want e domain boundaries...

Potentially invalid LTV records:
Empty DataFrame
Columns: [Loan_to_Value, Loan_Amount, Collateral_Value]
Index: []

Number of LTV domain violations:
0


### 📊 Domain Boundary Validation Results: `Loan_to_Value`

#### 🔍 Summary of Findings
* **Target Boundary Range:** $0\% < \text{LTV} \le 100\%$
* **Violations Found:** `0` records out of total dataset rows.

#### 💡 Interpretation & Action
The verification confirms that **all `Loan_to_Value` figures strictly adhere to logical financial limits**. There are no negative values, zero-value anomalies, or over-collateralization edge cases exceeding 100% in the dataset. 

Since no operational errors or impossible ratios were detected, **no row deletion or clipping is required** for `Loan_to_Value`. The feature is clean, domain-compliant, and ready for model training.

## Investigation 2: Credit Score

### Strategy & Validation Framework
`Credit_Score` is a strictly bounded financial metric governed by standard credit bureau scoring models (such as FICO or VantageScore). 

* **Valid Domain Bounds:** $[300, 850]$
* **Validation Strategy:**: **Domain Boundary Validation**. 
* **Objective:** Verify that every recorded score falls within $300$ and $850$. Any value below $300$ or above $850$ represents a corrupt data entry or operational error that must be handled.

In [14]:
print("Credit Score Range Check")
print("------------------------")

print("Minimum Credit Score:", X_train["Credit_Score"].min())
print("Maximum Credit Score:", X_train["Credit_Score"].max())

# Check values outside the valid range
invalid_scores = X_train[
    (X_train["Credit_Score"] < 300) |
    (X_train["Credit_Score"] > 850)
]

print("\nNumber of invalid Credit Scores:", len(invalid_scores))

print("\nInvalid Credit Scores:")
print(invalid_scores["Credit_Score"])

Credit Score Range Check
------------------------
Minimum Credit Score: 300.0
Maximum Credit Score: 900.0

Number of invalid Credit Scores: 9

Invalid Credit Scores:
2775     897.0
11510    900.0
14610    900.0
16633    866.0
6258     860.0
3297     900.0
7656     851.0
19542    852.0
4334     900.0
Name: Credit_Score, dtype: float64


### 📊 Findings & Remediation Strategy

#### Summary of Results
* **Observed Minimum Score:** `300.0` *(Valid lower bound)*
* **Observed Maximum Score:** `900.0` *(Exceeds valid upper bound)*
* **Invalid Records Identified:** **9 records** strictly exceed the $850$ credit bureau threshold (ranging from $851.0$ to $900.0$).

---

#### 🔍 Analysis & Explanation
The minimum score ($300.0$) complies perfectly with domain limits, but **9 observations violate the maximum ceiling of $850$**. 

These values represent clear **data entry errors or system artifacts** (possibly capped at 900 due to a non-standard scoring scale used upstream). Because these 11 rows violate hard real-world constraints, they cannot be accepted as valid credit scores in their current form.

---

#### 🛠️ Recommended Action
Since 9 records represent a tiny fraction ($<0.1\%$) of the dataset, the chosen action is:

 **Clip / Cap at Max Bound (Preferred):** Cap these 9 values at the standard maximum limit of `850.0` (apply `X_train['Credit_Score'] = X_train['Credit_Score'].clip(upper=850.0)` and the same fixed rule to `X_test`). This preserves the underlying applicant profile while bringing the feature into valid domain compliance.




In [15]:
# Cap the fixed credit-score boundary in both datasets.
# This uses a domain rule, not a statistic learned from the test set.
for dataset in (X_train, X_test):
    dataset.loc[dataset["Credit_Score"] > 850, "Credit_Score"] = 850

print("Maximum X_train Credit Score after correction:", X_train["Credit_Score"].max())
print("Maximum X_test Credit Score after correction:", X_test["Credit_Score"].max())

Maximum X_train Credit Score after correction: 850.0
Maximum X_test Credit Score after correction: 850.0


## Investigation 3: Years at Current Job


### Strategy & Validation Framework
`Years_at_Current_Job` is a tenure metric bounded by human employment plausibility. While statistical IQR rules flagged $736$ extreme values for this feature, IQR fails to consider real-world context—a long tenure at a single company is legitimate behavior, not automatically an error.

* **Valid Domain Bounds:** $[0, 50]$ years (based on standard working-age caps).
* **Validation Strategy:** Apply **Domain Boundary Validation** to inspect the minimum and maximum observed values to ensure they fall within realistic human limits.
* **Objective:** Verify that no impossible negative values or non-human upper limits (e.g., $70+$ years at a single company) exist in the data.

In [16]:
print("Minimum years at current job:", X_train["Years_at_Current_Job"].min())
print("Maximum years at cuurent job:", X_train["Years_at_Current_Job"].max())

Minimum years at current job: 0
Maximum years at cuurent job: 42


### 📊 Findings & Conclusion

#### Summary of Results
* **Observed Minimum Tenure:** `0` years *(Valid lower bound for new hires)*
* **Observed Maximum Tenure:** `42` years *(Plausible upper bound for long-tenured employees)*
* **Domain Violations Identified:** **0 records**

---

#### 💡 Interpretation
Even though statistical IQR flagged $736$ values as "outliers," domain validation confirms that the maximum tenure of **$42$ years** is entirely realistic for a senior worker or late-career applicant. There are no negative numbers or impossible career spans.

Because all values strictly abide by real-world human bounds, **no rows need to be removed, clipped, or altered**. The statistical "outliers" are legitimate extreme observations that reflect true tenure diversity.

## Investigation 4: Total Work Experience 

### 🔍 Investigation 4: Total Work Experience

### Strategy & Validation Framework
`Total_Work_Experience` measures an applicant's cumulative career duration. Like job tenure, statistical IQR rules flagged $466$ potential "outliers" here, but high work experience naturally occurs among mid- to late-career professionals.

* **Valid Domain Bounds:** $[0, 50]$ years (based on standard career lengths starting from working age $\sim18–25$ up to retirement $\sim65–70$).
* **Validation Strategy:** Apply **Domain Boundary Validation** to verify that career lengths fall within plausible human working-life limits.
* **Objective:** Ensure there are no negative experience figures or impossible values (e.g., $60+$ years of experience) that indicate data corruption.

In [17]:
print("Minimum total work experience:", X_train["Total_Work_Experience"].min())
print("Maximum total work experience:", X_train["Total_Work_Experience"].max())
 

Minimum total work experience: 0
Maximum total work experience: 44


### 📊 Findings & Conclusion

#### Summary of Results
* **Observed Minimum Experience:** `0` years *(Valid lower bound for entry-level applicants)*
* **Observed Maximum Experience:** `44` years *(Plausible upper bound for late-career professionals)*
* **Domain Violations Identified:** **0 records**

---

#### 💡 Interpretation
The maximum observed experience of **$44$ years** fits perfectly within a standard $40\text{–}45$ year working lifespan (e.g., an applicant who entered the workforce at age $20$ and is now $64$). 

Although IQR flagged $466$ observations as statistical outliers due to the right-skewed nature of career experience, domain validation proves that **all values are physically and logically realistic**. No rows need to be filtered, clipped, or modified.

## 🔗 Joint Relational Check: `Years_at_Current_Job` vs. `Total_Work_Experience`

### Strategy & Validation Framework
Checking features in isolation is not enough. Individual values like $10$ years at current job and $5$ years total experience look fine on their own, but when paired together, they create a mathematical impossibility.

* **Target Features:** `Years_at_Current_Job` AND `Total_Work_Experience` (Joint Evaluation)
* **Validation Constraint:** $\text{Years\_at\_Current\_Job} \le \text{Total\_Work\_Experience}$
* **Objective:** Verify cross-feature harmony. If a violation occurs, it signals a data entry error in **at least one** of these two interconnected columns.

In [18]:
print("Checking relational consistency...")
print()

relational_mismatch = X_train.loc[
    X_train["Years_at_Current_Job"] > X_train["Total_Work_Experience"],
    [
        "Years_at_Current_Job",
        "Total_Work_Experience"
    ]
]

print("Relational mismatches:")
print(relational_mismatch)

print("\nNumber of relational violations:")
print(len(relational_mismatch))

Checking relational consistency...

Relational mismatches:
Empty DataFrame
Columns: [Years_at_Current_Job, Total_Work_Experience]
Index: []

Number of relational violations:
0


### 📊 Findings & Conclusion

#### Summary of Results
* **Joint Constraint Tested:** $\text{Years\_at\_Current\_Job} \le \text{Total\_Work\_Experience}$
* **Cross-Feature Violations:** **0 records**

---

#### 💡 Interpretation
Zero violations were detected across the entire dataset. This confirms that both employment metrics are mutually consistent—no applicant has a current tenure that exceeds their total career duration. 

This joint validation confirms that both `Years_at_Current_Job` and `Total_Work_Experience` are not only individually realistic, but also **fully aligned with each other**. Both features are validated and ready for model training without modification.

  ## Investigation 5: Credit_Card_Utilization

### Strategy & Validation Framework
`Credit_Card_Utilization` represents the ratio of an applicant's used credit relative to their total limit. While statistical IQR rules flagged $377$ observations as potential outliers, percentage ratios are governed by standard financial boundaries.

* **Valid Domain Bounds:** $[0\%, 100\%]$
* **Validation Strategy:** Enforce **Domain Boundary Validation** to inspect for impossible negative utilization ratios or severe over-limit values.
* **Objective:** Ensure no corrupt data entries ($< 0\%$) or system artifacts ($> 100\%$) exist in the feature before feeding it to downstream models.

In [19]:
print("Minimum credit card utilization:", X_train["Credit_Card_Utilization"].min())
print("Maximum credit card utilization:", X_train["Credit_Card_Utilization"].max())
 

Minimum credit card utilization: 0.0
Maximum credit card utilization: 98.98784180197512


### 📊 Findings & Conclusion

#### Summary of Results
* **Observed Minimum Utilization:** `0.0%` *(Valid lower bound)*
* **Observed Maximum Utilization:** `98.99%` *(Valid upper bound)*
* **Domain Violations Identified:** **0 records**

---

#### 💡 Interpretation
The observed minimum of `0.0%` represents applicants using none of their available credit, while the maximum of `98.99%` represents maxed-out borrowers operating just under their credit limits. 

Although statistical IQR rules flagged $377$ observations as outliers due to right-skewness, domain validation proves that **100% of the records sit strictly within the valid financial range ($0\% \le \text{Utilization} \le 100\%$)**. There are no negative balance anomalies or impossible over-limit ratios. 

High-utilization borrowers provide a crucial risk signal for credit decisioning. Because all values represent legitimate financial behavior within valid domain bounds, **no row deletion, clipping, or transformation is required**.

## ⚙️ Encoding Categorical Variables

### ⚙️ Strategy & Method Selection

Machine learning algorithms require numerical inputs to process features effectively. To convert our categorical features into numerical representations without introducing artificial bias, we select the encoding technique based on feature type:

---

#### 1. Binary Encoding (Target & Two-State Features)

Binary encoding is applied when a categorical variable contains exactly two unique classes:

* **Application:** Used primarily for binary features or target variables (e.g., mapping `Loan_Status` as `Approved → 1` and `Rejected → 0`).
* **Conceptual Meaning:** The assigned integers ($0$ and $1$) serve strictly as binary indicators to represent two distinct classes, rather than implying an ordinal hierarchy or rank.

---

#### 2. Ordinal Encoding (Ordered Categories)

Ordinal encoding is applied when categorical variables possess a clear, inherent hierarchy or sequence:

* **Application:** Features with a natural progression (e.g., mapping `Education` as `Primary → 0`, `Secondary → 1`, `Tertiary → 2`, `Postgraduate → 3`).
* **Conceptual Meaning:** Unlike binary encoding, the assigned integers explicitly preserve order ($0 < 1 < 2 < 3$). This communicates to the model that a higher integer corresponds to a higher level within that feature.

---

#### 3. Label Encoding vs. One-Hot Encoding (Nominal Categories)

When dealing with nominal features (categories without a natural order, such as `Marital_Status` or `Employment_Type`), choosing the right transformation is critical:

* **Label Encoding:** Converts nominal categories into arbitrary integers ($0, 1, 2, \dots$). While computationally compact, models cannot inherently tell that these numbers are arbitrary labels. Algorithms may misinterpret them as sequential ($0 < 1 < 2$), introducing artificial bias.
* **One-Hot Encoding (OHE):** Converts nominal categories into separate binary indicator columns ($0$ or $1$). OHE is preferred for nominal features because it completely eliminates unintended ordinal assumptions.

> **Key Takeaway:** Ordinal encoding uses integers to preserve a true category order, whereas One-Hot Encoding is used for unordered categories to prevent algorithms from assuming an artificial rank.

---

#### 4. Statistical Consideration: Multicollinearity Control

* **Dummy Variable Trap:** Multicollinearity control is mainly relevant to linear models. This project uses a Decision Tree, so the pipeline keeps all one-hot columns and does not drop a reference category.

---

#### 5. Encoding Mapping Plan

We define explicit numerical mappings for binary and ordinal features, and apply One-Hot Encoding without reference dropping to nominal features inside the reusable pipeline.

### 🔎 Step 1: Identifying Categorical Features

Before selecting an encoding strategy, we must audit the dataset to isolate all object and categorical columns.

* **Objective:** Automatically detect and list all categorical features currently stored as strings or categorical data types.
* **Purpose:** This inspection confirms which columns require numerical conversion before model training and helps us distinguish **ordinal** features (which have an inherent rank) from **nominal** features (which require one-hot encoding).

In [20]:
categorical_columns = X_train.select_dtypes(
    include=["str", "category"]
).columns

print("Categorical columns:")
print(categorical_columns)

print("\nNumber of categorical columns:", len(categorical_columns))

Categorical columns:
Index(['Marital_Status', 'Education', 'Residence_Type', 'Employment_Type',
       'Tax_Return_Filed', 'Loan_Purpose', 'Collateral'],
      dtype='str')

Number of categorical columns: 7


### 📊 Inspection Results & Encoding Strategy

#### Summary of Identified Features
* **Total Categorical Input Columns Detected:** `7`
* **Features Isolated:** `Marital_Status`, `Education`, `Residence_Type`, `Employment_Type`, `Tax_Return_Filed`, `Loan_Purpose`, and `Collateral`.

---



### 🔎 Step 2: Inspecting Feature Cardinality

Before applying any transformations, we inspect the number of unique values in each categorical column to choose the correct encoding strategy (e.g., binary mapping for $2$ unique values, ordinal encoding for ordered categories, or one-hot encoding for multi-category nominal features).

In [21]:
for col in categorical_columns:
    print(f"{col}: {X_train[col].nunique()} unique values")

Marital_Status: 4 unique values
Education: 6 unique values
Residence_Type: 2 unique values
Employment_Type: 5 unique values
Tax_Return_Filed: 2 unique values
Loan_Purpose: 6 unique values
Collateral: 2 unique values


#### 💡 Categorical Breakdown & Strategy

Based on the inspection of unique value counts across the 7 categorical input features, we categorize and assign encoding techniques as follows. `Loan_Status` was already separated into `y`, so it is not included in this feature scan:

1. **Target Variable (`Loan_Status` — 2 Unique Values):**
   * **Strategy:** Binary Encoding (`Approved → 1`, `Rejected → 0`).
   * **Rationale:** A standard two-class target variable requiring binary numerical labels for classification models.

2. **Binary / Two-State Features (`Residence_Type` — 2 Unique Values & `Collateral` — 2 Unique Values):**
   * **Strategy:** Binary Indicator / Mapping ($0/1$).
   * **Rationale:** Features with exactly two states do not require multi-column one-hot expansion; mapping directly to $0$ and $1$ keeps the dataset compact without introducing artificial rank.

3. **Ordinal Variable (`Education` — 6 Unique Values):**
   * **Strategy:** Ordinal Integer Mapping ($0, 1, 2, 3, 4, 5$).
   * **Rationale:** Contains 6 distinct tiers with an inherent hierarchical progression (e.g., *High School < Associate < Bachelor's < Master's < Doctorate < Post-Doc*). Sequential integers preserve this logical hierarchy.

4. **Multi-Category Nominal Variables (`Marital_Status` — 4 Unique Values, `Employment_Type` — 5 Unique Values, `Loan_Purpose` — 6 Unique Values):**
   * **Strategy:** One-Hot Encoding (`drop='first'`).
   * **Rationale:** These features possess multiple categories ($4$, $5$, and $6$ respectively) with no inherent ordering or rank. OHE creates binary dummy columns while dropping one reference category to prevent the dummy variable trap (multicollinearity).

5. **Flag Feature (`Tax_Return_Filed` — 3 Unique Values):**
   * **Strategy:** One-Hot Encoding (`drop='first'`) or Categorical Cleaning.
   * **Rationale:** Contains 3 unique values (e.g., `Yes`, `No`, and a potential `Missing`/`Unfiled` indicator). Treating it as a nominal feature ensures each state is captured without assuming sequential order.

### 🔎 Step 3: Inspecting Categorical Values & Cardinality

To select the precise encoding technique for each categorical column, we inspect both the total count of unique values and the specific category labels within every feature.

* **Objective:** Determine feature cardinality and identify underlying domain relationships (binary, ordinal, or nominal).


In [22]:
# Display number of categories and their names

for col in categorical_columns:
    print(f"\n===== {col} =====")
    print("Number of unique values:", X_train[col].nunique())
    print("Categories:", X_train[col].unique())


===== Marital_Status =====
Number of unique values: 4
Categories: <StringArray>
['Married', 'Single', 'Widowed', 'Divorced']
Length: 4, dtype: str

===== Education =====
Number of unique values: 6
Categories: <StringArray>
['Graduate', 'Post Graduate', 'High School', 'Diploma', 'No Formal', 'PhD']
Length: 6, dtype: str

===== Residence_Type =====
Number of unique values: 2
Categories: <StringArray>
['Urban', 'Rural']
Length: 2, dtype: str

===== Employment_Type =====
Number of unique values: 5
Categories: <StringArray>
['Self-Employed', 'Private', 'Skilled Labor', 'Government', 'Unemployed']
Length: 5, dtype: str

===== Tax_Return_Filed =====
Number of unique values: 2
Categories: <StringArray>
['Yes', 'No', nan]
Length: 3, dtype: str

===== Loan_Purpose =====
Number of unique values: 6
Categories: <StringArray>
['Vehicle', 'Business', 'Personal', 'Home', 'Education', 'Medical']
Length: 6, dtype: str

===== Collateral =====
Number of unique values: 2
Categories: <StringArray>
['Yes', 

### 📊 Category Inspection Results & Encoding Plan

Based on the observed categories across all $7$ categorical input features, we define the following transformations:

#### 1. Binary Features (Direct 0/1 Mapping)
* **`Residence_Type`:** `{'Rural': 0, 'Urban': 1}`
* **`Collateral`:** `{'No': 0, 'Yes': 1}`

---

#### 2. Ordinal Feature (Explicit Progression Mapping)
* **`Education`:** Mapped sequentially based on formal attainment level:
  `{'No Formal': 0, 'High School': 1, 'Diploma': 2, 'Graduate': 3, 'Post Graduate': 4, 'PhD': 5}`

---

#### 3. Nominal Features (One-Hot Encoding)
* **`Marital_Status` ($4$ categories):** `Single`, `Married`, `Widowed`, `Divorced`
* **`Employment_Type` ($5$ categories):** `Private`, `Government`, `Self-Employed`, `Unemployed`, `Skilled Labor`
* **`Tax_Return_Filed`:** observed values `Yes` and `No`; missing values become `Unknown` in the pipeline.
* **`Loan_Purpose` ($6$ categories):** `Vehicle`, `Home`, `Medical`, `Education`, `Business`, `Personal`

> **Note on `Tax_Return_Filed`:** `Unknown` is introduced by the pipeline's missing-value imputer before One-Hot Encoding; it is not an observed raw category.

## Feature Engineering

Feature engineering is the process of creating new, informative features from existing raw data to help machine learning algorithms uncover underlying patterns more effectively.

For our **Loan Approval Decision Tree**, we adhere to a purposeful feature creation strategy:

* **Domain-Driven Design:** We do not create features arbitrarily or purely to expand dataset width.
* **Logical Alignment:** Every engineered feature must share a clear, explainable relationship with applicant credit risk and loan approval probability.
* **Model Interpretability:** Quality features enhance tree splitting efficiency, leading to simpler, highly interpretable decision nodes.

### 🔎 Auditing Existing Features Prior to Engineering

Before constructing new indicators, we perform a thorough review of our cleaned, baseline feature set.

* **Baseline Inventory:** Reviewing all available numerical and categorical columns currently in the dataset.
* **Domain Relevance:** Identifying key financial metrics—such as income, debt obligations, assets, and loan terms—that can be combined into meaningful risk ratios.
* **Redundancy Avoidance:** Ensuring newly engineered variables directly complement, rather than duplicate, the signal already present in the existing features.

In [23]:
print("=========================================")
print("CURRENT TRAINING FEATURE SET")
print("=========================================")
print(X_train.columns.tolist())
print("\nNumber of X_train features:", X_train.shape[1])
print("Number of X_test features:", X_test.shape[1])
print("Feature schemas match:", X_train.columns.equals(X_test.columns))

CURRENT TRAINING FEATURE SET
['Age', 'Marital_Status', 'Education', 'Dependents', 'Residence_Type', 'City_Tier', 'Employment_Type', 'Years_at_Current_Job', 'Total_Work_Experience', 'Monthly_Income', 'Other_Income', 'Existing_Loans', 'Existing_Loan_Amount', 'Monthly_EMI', 'Debt_to_Income', 'Savings', 'Investments', 'Bank_Balance', 'Credit_Card_Utilization', 'Number_of_Bank_Accounts', 'Number_of_Credit_Cards', 'Credit_Score', 'Loan_Defaults', 'Missed_Payments', 'Tax_Return_Filed', 'Loan_Purpose', 'Loan_Amount', 'Loan_Tenure', 'Interest_Rate', 'Collateral', 'Collateral_Value', 'Loan_to_Value']

Number of X_train features: 32
Number of X_test features: 32
Feature schemas match: True


### 🛠️ Feature Engineering: Creating `Total_Debt_Exposure`

To give our model a clearer view of an applicant's total financial liabilities, we derive a consolidated liability metric though it is still possible for the model to figure this out on its own but we want to give the model the relationship directly instead of requiring the model to discover it from the two separate variables.

* **Rationale:** A borrower's risk profile depends on their total financial burden upon approval. Combining existing debt obligations with the requested principal provides a more realistic representation of their overall debt exposure.
* **Formula:**
  $$\text{Total\_Debt\_Exposure} = \text{Existing\_Loan\_Amount} + \text{Loan\_Amount}$$

In [24]:
# =========================================
# FEATURE ENGINEERING
# =========================================

# Create the same deterministic feature in both datasets.
for dataset in (X_train, X_test):
    dataset["Total_Debt_Exposure"] = (
        dataset["Existing_Loan_Amount"] + dataset["Loan_Amount"]
    )

print("=========================================")
print("FEATURE ENGINEERING")
print("=========================================")
print("New feature created: Total_Debt_Exposure")

print("\nSample calculation from X_train:")
print(X_train[[
    "Existing_Loan_Amount",
    "Loan_Amount",
    "Total_Debt_Exposure"
]].head(10))

print("\nX_train feature count:", X_train.shape[1])
print("X_test feature count:", X_test.shape[1])

FEATURE ENGINEERING
New feature created: Total_Debt_Exposure

Sample calculation from X_train:
       Existing_Loan_Amount  Loan_Amount  Total_Debt_Exposure
5164                      0      1396569              1396569
2385                      0       875653               875653
2160                      0      1757797              1757797
16852                     0      1242243              1242243
13383                     0      3133013              3133013
18309                     0      1632525              1632525
16378               3925394       803223              4728617
14532               7593250      2687507             10280757
5787                2504783       492057              2996840
3695                5123190      7000520             12123710

X_train feature count: 33
X_test feature count: 33


### 📊 Feature Engineering Verification & Statistics

The engineered feature `Total_Debt_Exposure` has been successfully computed and added to the dataset:

* **Column Addition:** The dataset width expanded by $+1$ column (increasing total features from $32$ to $33$).
* **Data Integrity:** Sample checks confirm that `Total_Debt_Exposure` accurately reflects the sum of `Existing_Loan_Amount` and `Loan_Amount` across all rows without introducing missing or NaN values.

# Pipeline-Based Preprocessing



## Define the ColumnTransformer and Preprocessing Pipeline

We now proceed to a workflow with a reusable scikit-learn preprocessing pipeline. The `ColumnTransformer` applies the correct transformation to each feature group while preserving all remaining numerical columns.

* **Numerical features:** Median imputation is applied to features whose missing values require a learned central value.
* **Zero-fill features:** `Other_Income` and `Investments` use `0`, because a missing value represents no reported amount.
* **Nominal features:** Categories are safely one-hot encoded with `handle_unknown="ignore"`.
* **Binary and ordinal features:** Domain-aware category orders preserve the mappings used in the learning section.

> **Important:** This pipeline is defined but deliberately not fitted in this notebook. During cross-validation, notebook 02 will fit a fresh copy on each training fold only.

In [25]:
# ============================================================
# COLUMNTRANSFORMER AND PREPROCESSING PIPELINE
# ============================================================
# ============================================================
# FEATURE GROUP DEFINITIONS
# ============================================================
MEDIAN_NUMERIC_FEATURES = [
    "Loan_to_Value",
    "Collateral_Value",
    "Bank_Balance",
    "Credit_Score",
    "Interest_Rate"
]

ZERO_FILL_FEATURES = ["Other_Income", "Investments"]

NOMINAL_FEATURES = [
    "Marital_Status",
    "Employment_Type",
    "Loan_Purpose"
]

BINARY_FEATURES = ["Residence_Type", "Collateral"]

EDUCATION_CATEGORIES = [[
    "No Formal",
    "High School",
    "Diploma",
    "Graduate",
    "Post Graduate",
    "PhD"
]]

# ============================================================
# PIPELINE FACTORY
# ============================================================
def create_preprocessing_pipeline():
    """Builds and returns the unfitted preprocessing pipeline."""
    
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    nominal_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        # FIXED: Removed drop="first" to allow handle_unknown="ignore" safely
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    tax_return_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    binary_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            categories=[["Rural", "Urban"], ["No", "Yes"]],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ])

    education_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            categories=EDUCATION_CATEGORIES,
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("median_numeric", numeric_pipeline, MEDIAN_NUMERIC_FEATURES),
            ("zero_fill", SimpleImputer(
                strategy="constant",
                fill_value=0
            ), ZERO_FILL_FEATURES),
            ("nominal", nominal_pipeline, NOMINAL_FEATURES),
            ("tax_return", tax_return_pipeline, ["Tax_Return_Filed"]),
            ("binary", binary_pipeline, BINARY_FEATURES),
            ("education", education_pipeline, ["Education"])
        ],
        remainder="passthrough"
    )

    return Pipeline(steps=[("preprocessor", preprocessor)])

### ✅ ColumnTransformer and Pipeline Summary

The preprocessing pipeline has been defined successfully and is now the single specification for feature preparation. It replaces the manual imputation, encoding, and category-mapping steps for the implemented workflow.

* **Consistency:** The same transformations will be applied to every training, validation, test, and future prediction record.
* **Unknown categories:** New nominal values will not interrupt prediction because the one-hot encoder is configured with `handle_unknown="ignore"`.
* **Leakage protection:** No statistics or categories have been learned yet. The pipeline will be fitted within cross-validation in notebook 02.

> **Next Step:** Add this unfitted preprocessing pipeline to the Decision Tree pipeline in the model-training notebook, then let cross-validation control all fitting operations.

In [26]:
#exporting raw Split Data
X_train.to_csv("../data/splits/X_train_raw.csv", index=False)
X_test.to_csv("../data/splits/X_test_raw.csv", index=False)
y_train.to_frame(name="Loan_Status").to_csv("../data/splits/y_train_raw.csv", index=False)
y_test.to_frame(name="Loan_Status").to_csv("../data/splits/y_test_raw.csv", index=False)
# Inspect all datasets
print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)



X_train shape: (16000, 33)
X_test shape: (4000, 33)
y_train shape: (16000,)
y_test shape: (4000,)
